In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("reconciliation_table", "oh_apm_stg.tmp.ctl_reconciliation_log_Prv")
dbutils.widgets.text("error_table", "oh_apm_stg.vendor_extracts.error_load_report_log_cpc_Stg_Prv")
dbutils.widgets.text("s3_path", "s3://gia-stg-oh-ue1-data-raw/haven/inbound/VE_EDW/weekly")

In [0]:
reconciliation_table = dbutils.widgets.get("reconciliation_table")
error_table = dbutils.widgets.get("error_table")
s3_path = dbutils.widgets.get("s3_path")

In [0]:
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, LongType

# ---------- Helper Functions ----------
def safe_get(task_key, key, default):
    try:
        return dbutils.jobs.taskValues.get(taskKey=task_key, key=key, debugValue=default)
    except Exception:
        return default

def safe_get_widget(widget_name, default_list=[]):
    try:
        raw = dbutils.widgets.get(widget_name)
        return [x.strip() for x in raw.split(",") if x.strip()]
    except Exception:
        return default_list

# ---------- Get values from Pre-validation ----------
gz_expected_info = safe_get("Pre_validation", "gz_expected_info", [])
gz_file_paths = safe_get("Pre_validation", "gz_file_paths", {})
missing_required_gz_files = safe_get("Pre_validation", "missing_required_gz_files", [])
start_load = safe_get("Pre_validation", "start_load", None)

# ---------- Get required files list from widget ----------
required_gz_list = safe_get_widget("required_gz_list")

# ---------- Set timestamps ----------
if not start_load:
    start_load = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
end_load = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# ---------- Determine Date_Received ----------
# Pick first existing file path to extract folder date
if gz_file_paths:
    first_path = list(gz_file_paths.values())[0]
    folder_name = first_path.split("/")[-2]
    date_received = folder_name.replace("dt=", "") if folder_name.startswith("dt=") else datetime.now().strftime("%Y-%m-%d")
else:
    date_received = datetime.now().strftime("%Y-%m-%d")

# ---------- Build Error Records ----------
error_records = []

# Combine all missing files: required missing + gz missing in S3
missing_files_set = set(missing_required_gz_files)  # start with required missing
missing_files_set.update([name for name, _ in gz_expected_info if name not in gz_file_paths])

# For every file in CTL, add a record
for file_name, _ in gz_expected_info:
    # Determine if missing
    is_missing = file_name in missing_files_set
    error_desc = ""
    if file_name in missing_required_gz_files:
        error_desc = f"MISSING REQUIRED FILE: {file_name}"
    elif file_name not in gz_file_paths:
        error_desc = f"MISSING FILE: {file_name}"
    
    error_records.append((
        file_name,
        "" if is_missing else date_received,
        start_load,
        end_load,
        0,
        error_desc
    ))

# Also include any required files that were not in CTL at all
ctl_files_set = set([name for name, _ in gz_expected_info])
for file_name in required_gz_list:
    if file_name not in ctl_files_set:
        error_records.append((
            file_name,
            "",
            start_load,
            end_load,
            0,
            f"MISSING REQUIRED FILE NOT IN CTL: {file_name}"
        ))

# ---------- Create DataFrame ----------
error_schema = StructType([
    StructField("File_Name", StringType(), True),
    StructField("Date_Received", StringType(), True),
    StructField("Start_Load_Date", StringType(), True),
    StructField("End_Load_Date", StringType(), True),
    StructField("Row_Number", LongType(), True),
    StructField("Error_Description", StringType(), True)
])

error_df = spark.createDataFrame(error_records, schema=error_schema)

# ---------- Write to Error Table ----------
if error_table:
    error_df.write.mode("append").saveAsTable(error_table)
    print(f"✅ Error report written to {error_table}")
    display(error_df)  # Show what was written
    # Set staging flag
    has_errors = error_df.filter(error_df.Error_Description != "").count() > 0
    dbutils.jobs.taskValues.set(key="staging_has_errors", value=has_errors)
else:
    print("⚠️ No error_table specified. Skipping write.")
